# 爆炸超压拟合公式优化 — Strategy A+D 组合

## 优化策略
### Strategy A：角度项 V 阶次从线性扩展到二次 (33参 → 45参)
原公式角度项只到 V 的一次方：`(c_i + c_j·V_n)·cos(kθ)`  
升级为：`(c_i + c_j·V_n + c_k·V_n²)·cos(kθ)` — 每个距离项新增 4 参数，共 +12 参数

### Strategy D：乘性速度修正因子 (45参 → 48参)
在完整公式外乘以方向性速度修正因子：  
`K_d = 1 + α·V_n·cos(θ) + β·V_n² + γ·V_n·cos(2θ)` — 新增 3 参数

**目标**：压低 Z=4 近场区域的最大相对误差（原最大 58%），同时保持整体 MAPE 不劣化。

In [ ]:
%% 第一步：读取数据与拟合高精度静爆基准公式 (15参数傅里叶解耦版)
disp('>>> 开始步骤1：提取高精度静爆本底 (15参数傅里叶解耦模型)...');

% 1. 读取数据
opts = detectImportOptions('zhuxing1.xlsx');
opts.VariableNamingRule = 'preserve';
df = readtable('zhuxing1.xlsx', opts);
df.Properties.VariableNames(1:6) = {'M', 'H', 'V', 'P_MPa', 'Z', 'theta'};

% 2. 预处理环境因子 (萨克斯定律)
T0 = 288.15;
df.T_h = T0 - 6.5 * df.H;
df.Sp = (df.T_h / T0) .^ 5.25588;

% 3. 提取静爆数据
df_static = df(df.V == 0, :);

% 4. 定义 15参数 高阶静爆模型
% 采用 cos(k*theta) k=1..4 捕捉 0度与180度不对称性，远/中/近场独立解耦
static_model = @(p, X) ...
    (X(:,2).^(2/3) ./ X(:,1))   .* ( p(1) + p(2).*cosd(X(:,3)) + p(3).*cosd(2*X(:,3)) + p(4).*cosd(3*X(:,3)) + p(5).*cosd(4*X(:,3)) ) ...
  + (X(:,2).^(1/3) ./ X(:,1).^2) .* ( p(6) + p(7).*cosd(X(:,3)) + p(8).*cosd(2*X(:,3)) + p(9).*cosd(3*X(:,3)) + p(10).*cosd(4*X(:,3)) ) ...
  + (1 ./ X(:,1).^3)             .* ( p(11) + p(12).*cosd(X(:,3)) + p(13).*cosd(2*X(:,3)) + p(14).*cosd(3*X(:,3)) + p(15).*cosd(4*X(:,3)) );

% 5. 初始化猜测值与求解器配置
X_stat = [df_static.Z, df_static.Sp, df_static.theta];
y_stat = df_static.P_MPa;

p0_stat = 0.01 * ones(1, 15);
p0_stat(1) = 0.065; p0_stat(6) = 0.397; p0_stat(11) = 0.322;  % 锚定经典经验主轴常数

options_stat = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'off', 'MaxFunctionEvaluations', 100000, 'MaxIterations', 5000);

[p_stat, ~] = lsqcurvefit(static_model, p0_stat, X_stat, y_stat, [], [], options_stat);
p_stat = round(p_stat, 4);  % 系数截断4位小数，确保公式可复现

% 提取静爆常数项，供第二步动爆初始化使用
A = p_stat(1); B = p_stat(6); C = p_stat(11);

% 6. 计算静爆误差指标
y_stat_calc = static_model(p_stat, X_stat);
R2_stat = 1 - sum((y_stat - y_stat_calc).^2) / sum((y_stat - mean(y_stat)).^2);
abs_err_stat = abs(y_stat_calc - y_stat);
rel_err_stat = (abs_err_stat ./ y_stat) * 100;
MAE_stat = mean(abs_err_stat);
MAPE_stat = mean(rel_err_stat);

fprintf('\n================ 第一步：静爆拟合完成 ================\n');
fprintf('R² = %.4f | MAE = %.5f MPa | MAPE = %.2f %%\n', R2_stat, MAE_stat, MAPE_stat);
fprintf('\n【静爆公式 (15参数)】\n');
fprintf('P_stat = (Sp^(2/3)/Z)   * [%.4f + %.4f*cosθ + %.4f*cos2θ + %.4f*cos3θ + %.4f*cos4θ]\n', p_stat(1:5));
fprintf('       + (Sp^(1/3)/Z^2) * [%.4f + %.4f*cosθ + %.4f*cos2θ + %.4f*cos3θ + %.4f*cos4θ]\n', p_stat(6:10));
fprintf('       + (1/Z^3)        * [%.4f + %.4f*cosθ + %.4f*cos2θ + %.4f*cos3θ + %.4f*cos4θ]\n', p_stat(11:15));
fprintf('=====================================================\n');


## 第二步A：拟合原33参数模型作为初始化基准

先用与原始 `gongshi2.ipynb` 完全相同的 33 参数模型拟合动爆数据，
为后续扩展到 45 参数结构提供稳健的初始猜测值。

In [ ]:
%% 第二步A：先拟合原33参数模型以获取稳健初始值
disp('>>> 步骤2A：拟合原33参数模型作为初始化基准...');

% 提取动爆数据
df_dyn = df(df.V > 0, :);
X_dyn = [df_dyn.Z, df_dyn.Sp, df_dyn.theta, df_dyn.V];
y_dyn = df_dyn.P_MPa;

% 原始33参数模型 (与 gongshi2.ipynb 完全一致)
% 每距离项11参数: 常数(1+Vn+Vn²) + cosθ(1+Vn) + cos2θ(1+Vn) + cos3θ(1+Vn) + cos4θ(1+Vn)
dyn_model_33 = @(c, X) ...
    (X(:,2).^(2/3) ./ X(:,1)) .* ( c(1) + c(2).*(X(:,4)/1000) + c(3).*(X(:,4)/1000).^2 + ...
        (c(4) + c(5).*(X(:,4)/1000)).*cosd(X(:,3)) + (c(6) + c(7).*(X(:,4)/1000)).*cosd(2*X(:,3)) + ...
        (c(8) + c(9).*(X(:,4)/1000)).*cosd(3*X(:,3)) + (c(10) + c(11).*(X(:,4)/1000)).*cosd(4*X(:,3)) ) ...
    + ...
    (X(:,2).^(1/3) ./ X(:,1).^2) .* ( c(12) + c(13).*(X(:,4)/1000) + c(14).*(X(:,4)/1000).^2 + ...
        (c(15) + c(16).*(X(:,4)/1000)).*cosd(X(:,3)) + (c(17) + c(18).*(X(:,4)/1000)).*cosd(2*X(:,3)) + ...
        (c(19) + c(20).*(X(:,4)/1000)).*cosd(3*X(:,3)) + (c(21) + c(22).*(X(:,4)/1000)).*cosd(4*X(:,3)) ) ...
    + ...
    (1 ./ X(:,1).^3) .* ( c(23) + c(24).*(X(:,4)/1000) + c(25).*(X(:,4)/1000).^2 + ...
        (c(26) + c(27).*(X(:,4)/1000)).*cosd(X(:,3)) + (c(28) + c(29).*(X(:,4)/1000)).*cosd(2*X(:,3)) + ...
        (c(30) + c(31).*(X(:,4)/1000)).*cosd(3*X(:,3)) + (c(32) + c(33).*(X(:,4)/1000)).*cosd(4*X(:,3)) );

% 初始猜测：锚定静爆常数项
c0_33 = 0.05 * ones(1, 33);
c0_33(1) = A; c0_33(12) = B; c0_33(23) = C;

options_33 = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'off', 'MaxFunctionEvaluations', 500000, 'MaxIterations', 5000);

[c33_opt, ~] = lsqcurvefit(dyn_model_33, c0_33, X_dyn, y_dyn, [], [], options_33);
c33_opt = round(c33_opt, 4);

% 基准33参数误差
P33_pred = dyn_model_33(c33_opt, X_dyn);
R2_33 = 1 - sum((y_dyn - P33_pred).^2) / sum((y_dyn - mean(y_dyn)).^2);
abs33 = abs(P33_pred - y_dyn);
rel33 = (abs33 ./ y_dyn) * 100;
MAPE_33 = mean(rel33);
max_err_33 = max(rel33);

fprintf('\n================ 基准33参数拟合完成 ================\n');
fprintf('R² = %.4f | MAE = %.5f MPa | MAPE = %.2f %% | 最大误差 = %.2f %%\n', ...
    R2_33, mean(abs33), MAPE_33, max_err_33);
fprintf('=====================================================\n');


## 第二步B：33参数 → 45参数结构映射

每个距离项从 11 参数扩展为 15 参数：
- 常数项保持不变: `c0 + c1·Vn + c2·Vn²`
- 角度项从 `ci + cj·Vn` 扩展为 `ci + cj·Vn + ck·Vn²`，新增的 V² 项初始化为 0

**映射表 (每个距离项内，0-indexed)**:
```
旧结构 idx: [0,  1,  2,  3,  4,  5,  6,  7,  8,  9,  10]
              ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓
新结构 idx: [0,  1,  2,  3,  4,  6,  7,  9, 10, 12, 13]
新增 idx:                       5        8      11      14  (初始化为0)
```

In [ ]:
%% 第二步B：33参数 → 45参数结构映射，构建48参数初始向量
disp('>>> 步骤2B：扩展为45参数结构 + 3参数修正因子...');

% 每个距离项内 11→15 的 0-indexed 位置映射
% 旧11参: [const0, constV, constV2, cos1_0, cos1_V, cos2_0, cos2_V,
%          cos3_0, cos3_V, cos4_0, cos4_V]
% 新15参: [const0, constV, constV2, cos1_0, cos1_V, cos1_V2, cos2_0, cos2_V, cos2_V2,
%          cos3_0, cos3_V, cos3_V2, cos4_0, cos4_V, cos4_V2]
% 映射: 旧位置(0-idx) → 新位置(0-idx)
map_old_to_new_0idx = [0, 1, 2, 3, 4, 6, 7, 9, 10, 12, 13];

c0_48 = zeros(1, 48);

for term = 0:2  % 遍历3个距离项
    for old_local_1idx = 1:11  % MATLAB 1-indexed 遍历旧项内11个参数
        new_local_0idx = map_old_to_new_0idx(old_local_1idx);  % 映射到新0-indexed位置
        old_idx = term * 11 + old_local_1idx;                 % 旧向量中的 MATLAB 索引
        new_idx = term * 15 + new_local_0idx + 1;             % 新向量中的 MATLAB 索引(+1转换)
        c0_48(new_idx) = c33_opt(old_idx);
    end
end

% 新增的 V² 角度项 (每距离项的位置 5, 8, 11, 14，0-indexed)
% 已由 zeros 初始化自动为0，无需额外操作

% 修正因子 c(46:48) 初始化为 0 (无修正状态)
% c0_48(46:48) 已经是 0

fprintf('✅ 48参数初始向量构建完成。\n');
fprintf('   基础45参数：继承33参数最优解，新增V²角度项=0\n');
fprintf('   修正3参数：初始化为0 (K_d = 1，无修正状态)\n');
fprintf('   验证: c0_48(1)=%.4f (应为A=%.4f), c0_48(16)=%.4f (应为B=%.4f), c0_48(31)=%.4f (应为C=%.4f)\n', ...
    c0_48(1), A, c0_48(16), B, c0_48(31), C);


## 第二步C：定义并拟合 48参数 A+D 组合模型

### 模型结构
```
P_dyn = [Term1(Z,Sp,θ,V) + Term2(Z,Sp,θ,V) + Term3(Z,Sp,θ,V)] × K_d(V,θ)

Term_i = D_i(Z,Sp) × Σ_{k=0}^{4} (a_{i,k} + b_{i,k}·Vn + c_{i,k}·Vn²) × cos(kθ)
K_d    = 1 + α·Vn·cosθ + β·Vn² + γ·Vn·cos(2θ)
```

其中 D₁ = Sp^(2/3)/Z, D₂ = Sp^(1/3)/Z², D₃ = 1/Z³, Vn = V/1000

In [ ]:
%% 第二步C：定义48参数 A+D 组合模型并拟合
disp('>>> 步骤2C：拟合48参数 A+D 组合模型...');
disp('    (此步骤可能需要数分钟，请耐心等待...)');

% ===== Strategy A: 45参数基础模型 (角度项含V²) =====
% 每距离项15参数排列:
%   [const0, constV, constV2,                                                % 常数项(3)
%    cos1_0, cos1_V, cos1_V2, cos2_0, cos2_V, cos2_V2,                      % cosθ, cos2θ (各3)
%    cos3_0, cos3_V, cos3_V2, cos4_0, cos4_V, cos4_V2]                      % cos3θ, cos4θ (各3)

model_A_45 = @(c, Sp, Z, theta, Vn) ...
    (Sp.^(2/3) ./ Z)     .* ( c(1)  + c(2).*Vn  + c(3).*Vn.^2 + ...
        (c(4)  + c(5).*Vn  + c(6).*Vn.^2) .* cosd(theta) + ...
        (c(7)  + c(8).*Vn  + c(9).*Vn.^2) .* cosd(2*theta) + ...
        (c(10) + c(11).*Vn + c(12).*Vn.^2) .* cosd(3*theta) + ...
        (c(13) + c(14).*Vn + c(15).*Vn.^2) .* cosd(4*theta) ) ...
  + (Sp.^(1/3) ./ Z.^2)  .* ( c(16) + c(17).*Vn + c(18).*Vn.^2 + ...
        (c(19) + c(20).*Vn + c(21).*Vn.^2) .* cosd(theta) + ...
        (c(22) + c(23).*Vn + c(24).*Vn.^2) .* cosd(2*theta) + ...
        (c(25) + c(26).*Vn + c(27).*Vn.^2) .* cosd(3*theta) + ...
        (c(28) + c(29).*Vn + c(30).*Vn.^2) .* cosd(4*theta) ) ...
  + (1 ./ Z.^3)           .* ( c(31) + c(32).*Vn + c(33).*Vn.^2 + ...
        (c(34) + c(35).*Vn + c(36).*Vn.^2) .* cosd(theta) + ...
        (c(37) + c(38).*Vn + c(39).*Vn.^2) .* cosd(2*theta) + ...
        (c(40) + c(41).*Vn + c(42).*Vn.^2) .* cosd(3*theta) + ...
        (c(43) + c(44).*Vn + c(45).*Vn.^2) .* cosd(4*theta) );

% ===== Strategy D: 乘性速度修正因子 =====
% K_d = 1 + c(46)*Vn*cos(θ) + c(47)*Vn² + c(48)*Vn*cos(2θ)
% cos(θ) 项: 前向增强 (0°最强, 90°无效, 180°反向)
% Vn² 项:   通用高速非线性修正 (所有角度均等)
% cos(2θ)项: 前/后向双峰增强 (0°和180°最强, 90°反向)

full_model_48 = @(c, X) ...
    model_A_45(c(1:45), X(:,2), X(:,1), X(:,3), X(:,4)/1000) .* ...
    (1 + c(46).*(X(:,4)/1000).*cosd(X(:,3)) + c(47).*(X(:,4)/1000).^2 + c(48).*(X(:,4)/1000).*cosd(2*X(:,3)));

% 拟合配置 (增大评估次数和迭代上限以应对48维参数空间)
options_48 = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'iter-detailed', 'MaxFunctionEvaluations', 300000, 'MaxIterations', 10000, ...
    'FunctionTolerance', 1e-10, 'OptimalityTolerance', 1e-10, ...
    'StepTolerance', 1e-10);

t_start = tic;
[c48_opt, resnorm, residual, exitflag, output] = ...
    lsqcurvefit(full_model_48, c0_48, X_dyn, y_dyn, [], [], options_48);
elapsed = toc(t_start);

% 系数截断4位小数
c48_opt = round(c48_opt, 4);

fprintf('\n================ 48参数A+D模型拟合完成 ================\n');
fprintf('耗时: %.1f 秒 | exitflag = %d | 残差平方和: %.6f\n', elapsed, exitflag, resnorm);
fprintf('迭代次数: %d | 函数评估次数: %d\n', output.iterations, output.funcCount);
fprintf('========================================================\n');


In [ ]:
%% 第二步D：计算48参数模型预测值与误差指标
disp('>>> 步骤2D：计算预测误差...');

% 预测值
P_dyn_pred = full_model_48(c48_opt, X_dyn);
df_dyn.P_pred = P_dyn_pred;

% 整体误差指标
R2_dyn = 1 - sum((y_dyn - P_dyn_pred).^2) / sum((y_dyn - mean(y_dyn)).^2);
abs_err_dyn = abs(y_dyn - P_dyn_pred);
rel_err_dyn = (abs_err_dyn ./ y_dyn) * 100;
MAE_dyn = mean(abs_err_dyn);
MAPE_dyn = mean(rel_err_dyn);
max_err_dyn = max(rel_err_dyn);

fprintf('\n================ 48参数 A+D 模型误差评估 ================\n');
fprintf('整体指标:\n');
fprintf('  R²        = %.4f\n', R2_dyn);
fprintf('  MAE       = %.5f MPa\n', MAE_dyn);
fprintf('  MAPE      = %.2f %%\n', MAPE_dyn);
fprintf('  最大误差  = %.2f %%\n', max_err_dyn);
fprintf('\n');

% 分Z段统计
fprintf('分距离段统计:\n');
fprintf('  Z    | 样本数 | MAPE%%  | 最大%%  | 90%%分位数%%\n');
fprintf('  -----|--------|--------|--------|----------\n');
for z_val = sort(unique(df_dyn.Z))'
    mask_z = (df_dyn.Z == z_val);
    err_z = rel_err_dyn(mask_z);
    mape_z = mean(err_z);
    max_z  = max(err_z);
    p90_z  = prctile(err_z, 90);
    fprintf('  Z=%.0f  |  %4d  | %6.2f | %6.2f | %8.2f\n', z_val, sum(mask_z), mape_z, max_z, p90_z);
end
fprintf('\n');

% Z=4 分角度详细排查 (关键问题区域)
mask_z4 = (df_dyn.Z == 4);
fprintf('Z=4 分角度误差 (关键区域):\n');
fprintf('  θ°   | MAPE%%  | 最大%%  | P_pred/P_sim\n');
fprintf('  -----|--------|--------|-----------\n');
for th = sort(unique(df_dyn.theta))'
    mask_th = mask_z4 & (df_dyn.theta == th);
    if sum(mask_th) > 0
        err_th = rel_err_dyn(mask_th);
        ratio_th = mean(P_dyn_pred(mask_th) ./ y_dyn(mask_th));
        fprintf('  θ=%3.0f°| %6.2f | %6.2f | %8.3f\n', th, mean(err_th), max(err_th), ratio_th);
    end
end
fprintf('============================================================\n');


In [ ]:
%% 第二步E：输出48参数 A+D 公式全貌
fprintf('\n');
fprintf('╔══════════════════════════════════════════════════════════════╗\n');
fprintf('║           48参数 A+D 动爆公式完整系数表                      ║\n');
fprintf('╠══════════════════════════════════════════════════════════════╣\n');
fprintf('║  P_dyn = [Term1 + Term2 + Term3] × K_d                      ║\n');
fprintf('║  其中: V_n = V / 1000, Sp = (T_h/T0)^5.25588               ║\n');
fprintf('╠══════════════════════════════════════════════════════════════╣\n');
fprintf('║                                                              ║\n');

fprintf('║  【Term1】= (Sp^(2/3)/Z) × [                                ║\n');
fprintf('║    常数:  (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(1:3));
fprintf('║    cosθ:  (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(4:6));
fprintf('║    cos2θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(7:9));
fprintf('║    cos3θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(10:12));
fprintf('║    cos4θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn² ]              ║\n', c48_opt(13:15));
fprintf('║                                                              ║\n');

fprintf('║  【Term2】= (Sp^(1/3)/Z²) × [                               ║\n');
fprintf('║    常数:  (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(16:18));
fprintf('║    cosθ:  (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(19:21));
fprintf('║    cos2θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(22:24));
fprintf('║    cos3θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(25:27));
fprintf('║    cos4θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn² ]              ║\n', c48_opt(28:30));
fprintf('║                                                              ║\n');

fprintf('║  【Term3】= (1/Z³) × [                                      ║\n');
fprintf('║    常数:  (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(31:33));
fprintf('║    cosθ:  (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(34:36));
fprintf('║    cos2θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(37:39));
fprintf('║    cos3θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn²                ║\n', c48_opt(40:42));
fprintf('║    cos4θ: (%+.4f) + (%+.4f)·Vn + (%+.4f)·Vn² ]              ║\n', c48_opt(43:45));
fprintf('║                                                              ║\n');

fprintf('║  【K_d 乘性修正因子】                                       ║\n');
fprintf('║    K_d = 1 + (%+.4f)·Vn·cosθ + (%+.4f)·Vn² + (%+.4f)·Vn·cos2θ ║\n', c48_opt(46:48));
fprintf('║                                                              ║\n');
fprintf('╚══════════════════════════════════════════════════════════════╝\n');


## 第三步：新旧模型全面对比分析

In [ ]:
%% 第三步：原始33参数 vs A+D 48参数 全面对比
disp('>>> 步骤3：新旧模型对比分析...');

fprintf('\n');
fprintf('╔══════════════════════════════════════════════════════════════╗\n');
fprintf('║          原始33参  vs  A+D 48参  性能对比                    ║\n');
fprintf('╠══════════════════╦═════════════════╦════════════════════════╣\n');
fprintf('║      指标        ║   原33参数      ║   A+D 48参数          ║\n');
fprintf('╠══════════════════╬═════════════════╬════════════════════════╣\n');
fprintf('║ R²               ║   %.4f        ║   %.4f              ║\n', R2_33, R2_dyn);
fprintf('║ MAE (MPa)        ║   %.5f        ║   %.5f              ║\n', mean(abs33), MAE_dyn);
fprintf('║ MAPE (%%)         ║   %.2f         ║   %.2f               ║\n', MAPE_33, MAPE_dyn);
fprintf('║ 最大误差 (%%)     ║   %.2f         ║   %.2f               ║\n', max_err_33, max_err_dyn);
fprintf('║ 90%%分位数误差(%%) ║   %.2f         ║   %.2f               ║\n', ...
    prctile(rel33, 90), prctile(rel_err_dyn, 90));
fprintf('║ 95%%分位数误差(%%) ║   %.2f         ║   %.2f               ║\n', ...
    prctile(rel33, 95), prctile(rel_err_dyn, 95));
fprintf('╚══════════════════╩═════════════════╩════════════════════════╝\n');

% 分Z段最大误差对比
fprintf('\n分Z段最大误差对比:\n');
fprintf('  Z    | 原33最大%%  | A+D48最大%% | 改善量\n');
fprintf('  -----|------------|------------|-------\n');
for z_val = sort(unique(df_dyn.Z))'
    mask_z = (df_dyn.Z == z_val);
    m33 = max(rel33(mask_z));
    m48 = max(rel_err_dyn(mask_z));
    impr = m33 - m48;
    fprintf('  Z=%.0f  |   %6.2f    |   %6.2f    | %+.2f\n', z_val, m33, m48, impr);
end

% Z=4 分角度最大误差对比 (关键问题区域)
mask_z4 = (df_dyn.Z == 4);
fprintf('\nZ=4 分角度最大误差对比 (关键区域):\n');
fprintf('  θ°   | 原33最大%%  | A+D48最大%% | 改善量\n');
fprintf('  -----|------------|------------|-------\n');
for th = sort(unique(df_dyn.theta))'
    mask_th = mask_z4 & (df_dyn.theta == th);
    if sum(mask_th) > 0
        m33 = max(rel33(mask_th));
        m48 = max(rel_err_dyn(mask_th));
        impr = m33 - m48;
        fprintf('  θ=%3.0f°|   %6.2f    |   %6.2f    | %+.2f\n', th, m33, m48, impr);
    end
end

% 改善总结
fprintf('\n============= 改善总结 =============\n');
fprintf('最大误差: %.2f%% → %.2f%% (改善 %.1f 个百分点)\n', max_err_33, max_err_dyn, max_err_33 - max_err_dyn);
fprintf('MAPE:    %.2f%% → %.2f%% (%+.2f 个百分点)\n', MAPE_33, MAPE_dyn, MAPE_dyn - MAPE_33);
fprintf('R²:      %.4f → %.4f\n', R2_33, R2_dyn);
fprintf('=====================================\n');


## 第四步：导出误差分析 Excel

导出三个 Excel 文件：
1. **按工况排序** (H → V → Z → θ)：适合工程查阅
2. **按误差大小降序**：适合排查残差痛点
3. **静爆误差表**：静爆模型评估

In [ ]:
%% 第四步：导出误差分析 Excel (三个文件)
disp('>>> 步骤4：导出误差分析Excel文件...');

% ==================== 动爆误差表 ====================
% 构造输出表
df_out = df_dyn(:, {'M', 'H', 'V', 'Z', 'theta', 'P_MPa'});
df_out.P_pred = P_dyn_pred;
df_out.Abs_Error = abs_err_dyn;
df_out.Rel_Error_Pct = rel_err_dyn;
df_out.Error_Magnitude = abs(rel_err_dyn);  % 用于排序的辅助列

% 中文规范表头
df_out.Properties.VariableNames{'P_pred'} = 'P_拟合计算值_MPa';
df_out.Properties.VariableNames{'Abs_Error'} = '绝对误差_MPa';
df_out.Properties.VariableNames{'Rel_Error_Pct'} = '相对误差_百分比';
df_out.Properties.VariableNames{'Error_Magnitude'} = '误差绝对大小_用于排序';

% 版本A：按物理工况多级排序 (H → V → Z → theta)
df_by_condition = sortrows(df_out, {'H', 'V', 'Z', 'theta'});
filename_A = 'A+D_优化结果_按工况排序.xlsx';
writetable(df_by_condition, filename_A);
fprintf('✅ 表格A导出: 【%s】(适合编入查阅手册)\n', filename_A);

% 版本B：按相对误差大小降序排列
df_by_error = sortrows(df_out, '误差绝对大小_用于排序', 'descend');
filename_B = 'A+D_优化结果_按误差大小排序.xlsx';
writetable(df_by_error, filename_B);
fprintf('✅ 表格B导出: 【%s】(适合排查残差痛点)\n', filename_B);

% ==================== 静爆误差表 ====================
df_static_out = df_static(:, {'M', 'H', 'Z', 'theta', 'P_MPa'});
df_static_out.P_pred = static_model(p_stat, [df_static.Z, df_static.Sp, df_static.theta]);
abs_err_s = abs(df_static_out.P_pred - df_static_out.P_MPa);
rel_err_s = (abs_err_s ./ df_static_out.P_MPa) * 100;
df_static_out.Abs_Error = abs_err_s;
df_static_out.Rel_Error_Pct = rel_err_s;
df_static_out.Error_Magnitude = abs(rel_err_s);

df_static_out.Properties.VariableNames{'P_pred'} = 'P_拟合计算值_MPa';
df_static_out.Properties.VariableNames{'Abs_Error'} = '绝对误差_MPa';
df_static_out.Properties.VariableNames{'Rel_Error_Pct'} = '相对误差_百分比';
df_static_out.Properties.VariableNames{'Error_Magnitude'} = '误差绝对大小_用于排序';

df_static_sorted = sortrows(df_static_out, '误差绝对大小_用于排序', 'descend');
filename_S = 'A+D_优化结果_静爆误差表.xlsx';
writetable(df_static_sorted, filename_S);
fprintf('✅ 表格C导出: 【%s】(静爆模型误差)\n', filename_S);

fprintf('\n🎉 A+D 组合优化全部完成！所有结果已导出至当前文件夹。\n');


## 公式总结

### 最终动爆公式 (48参数 A+D 组合)

$$P_{dyn} = \left[T_1 + T_2 + T_3\right] \times K_d$$

$$T_i = D_i \cdot \sum_{k=0}^{4} (a_{i,k} + b_{i,k} V_n + c_{i,k} V_n^2) \cos(k\theta)$$

$$K_d = 1 + \alpha \cdot V_n \cdot \cos\theta + \beta \cdot V_n^2 + \gamma \cdot V_n \cdot \cos(2\theta)$$

其中 $D_1 = \frac{Sp^{2/3}}{Z}$, $D_2 = \frac{Sp^{1/3}}{Z^2}$, $D_3 = \frac{1}{Z^3}$, $V_n = V/1000$

### 策略要点
| 策略 | 改动 | 目的 |
|------|------|------|
| A | 角度项 V 阶次 1 → 2 | 捕获高V下角分布非线性 |
| D | 新增乘性修正 $K_d$ | 方向性前向增强+通用高速修正 |
| 初始化 | 33参→45参渐进扩展 | 确保优化从良好局部极小出发 |